# Seattle Crime — Proximity Analysis

Explore SPD crime incidents **near any Seattle address**: enter an address, the
notebook geocodes it, filters every reported crime within a chosen radius, and
renders a dashboard:

- a **summary line** — incidents/year, how that compares to the **citywide average**
  for an area this size, % violent, and a **gun-related** count with uptick flag;
- **crimes per year** and **gun-related incidents per year** (by severity);
- **top offenses** — past-year **counts** plus a **trend** view (avg incidents/year
  over the 10 / 5 / 1-year windows, so you can see what's rising);
- **by hour of day** — past 5 years vs past year overlaid (% of incidents, smoothed);
- and a spatial comparison of the **past 5 years vs past year**: a
  **proportional-symbol location map** (dot size = incidents/year) and a
  **per-year density heatmap**.

See [`README.md`](README.md) for setup (`uv sync`) and project details.

**Run the cells top to bottom:**

1. **Load** — auto-downloads the SPD CSV into `data/` on first run, then
   loads it (skipped if `df` is already in the kernel). As of June 2026 this is more than 337 MB and it will keep growing.
2. **Filter** — keep only the last `YEARS_BACK` years (default 10; this also drops
   the `1900-01-01` sentinels), then clip coordinates to a Seattle bbox.
3. **Functions** — define `geocode`, distance, and `analyze` helpers.
4. **Run** — running the last cell prompts you for an address (type or paste;
   blank = Space Needle). The dashboard renders inline and is saved to
   `charts/<address>.png`.

To analyze another address, just re-run that last cell and enter a new one.


In [ ]:
import pandas as pd
import urllib.request
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
# Socrata bulk CSV export for SPD Crime Data, 2008-Present (dataset tazs-3rd5)
DATASET_URL = "https://data.seattle.gov/api/views/tazs-3rd5/rows.csv?accessType=DOWNLOAD"

# Use any existing SPD CSV in data/; otherwise download it once (~400MB).
existing = sorted(DATA_DIR.glob("SPD_Crime_Data*.csv"))
if existing:
    csv_path = existing[-1]
else:
    csv_path = DATA_DIR / "SPD_Crime_Data__2008-Present.csv"
    print(f"No CSV found in {DATA_DIR}/ - downloading ~400MB (one time)...")

    # The server omits Content-Length (chunked/gzip), so report MB downloaded,
    # printing only every ~25 MB to avoid flooding the notebook output.
    _last = [0]

    def _progress(block, block_size, total):
        mb = block * block_size / 1e6
        if mb - _last[0] >= 25:
            _last[0] = mb
            print(f"  downloaded {mb:,.0f} MB...")

    urllib.request.urlretrieve(DATASET_URL, csv_path, reporthook=_progress)
    print(f"saved -> {csv_path}")


def parse_datetimes(s: pd.Series) -> pd.Series:
    """Parse SPD datetime strings. The Socrata API export uses MM/DD/YYYY; older
    manual exports use 'YYYY Mon DD'. Try the API format, fall back if it misses."""
    parsed = pd.to_datetime(s, format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    if parsed.isna().mean() > 0.5:
        parsed = pd.to_datetime(s, format="%Y %b %d %I:%M:%S %p", errors="coerce")
    return parsed


# Skip the ~400MB read if df is already loaded in this kernel.
if "df" in globals():
    print(f"df already loaded ({len(df):,} rows) - skipping CSV read")
else:
    df = pd.read_csv(csv_path, low_memory=False)
    for _col in ["Report DateTime", "Offense Date"]:
        df[_col] = parse_datetimes(df[_col])
    print(f"loaded {len(df):,} rows x {df.shape[1]} columns from {csv_path.name}")

df.head()

In [ ]:
# Keep only recent offenses. This also drops the sentinel 1900-01-01 dates
# (and any other pre-cutoff junk) since they fall outside the window.
YEARS_BACK = 10   # <-- easy to change: how many calendar years of history to include

# Snap to Jan 1 of (this year - YEARS_BACK) so the first year is complete
# (the current year is still partial since it isn't over yet).
cutoff = pd.Timestamp(year=pd.Timestamp.now().year - YEARS_BACK, month=1, day=1)
before = len(df)
df = df[df["Offense Date"] >= cutoff]
print(f"Keeping offenses since {cutoff:%Y-%m-%d}: "
      f"removed {before - len(df):,} rows; {len(df):,} remain")

In [ ]:
# --- Clean coordinates -------------------------------------------------
# Lat/Long are REDACTED for ~14.5% of rows and contain some junk geocodes
# (lat -1/90, lon +/-175). Coerce to numeric and clip to a Seattle bbox.
df["lat"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["lon"] = pd.to_numeric(df["Longitude"], errors="coerce")

SEATTLE_BBOX = dict(lat_min=47.48, lat_max=47.74, lon_min=-122.46, lon_max=-122.22)
in_seattle = (
    df["lat"].between(SEATTLE_BBOX["lat_min"], SEATTLE_BBOX["lat_max"])
    & df["lon"].between(SEATTLE_BBOX["lon_min"], SEATTLE_BBOX["lon_max"])
)
geo = df[in_seattle].copy()
print(f"{len(geo):,} rows have usable Seattle coordinates "
      f"({len(geo)/len(df)*100:.1f}% of {len(df):,})")

In [ ]:
# --- Analysis functions ------------------------------------------------
# geocode an address -> filter incidents within a radius -> draw + save charts.
# Called from the last cell via analyze(ADDRESS, RADIUS_MILES).
import re
import urllib.parse
import urllib.request
import json
import numpy as np
import matplotlib.pyplot as plt

SEATTLE_LAND_SQMI = 83.8  # for the citywide-density benchmark


def geocode(address: str) -> tuple[float, float]:
    """Geocode a free-form address via OpenStreetMap Nominatim (nudged to Seattle)."""
    query = address if "seattle" in address.lower() else f"{address}, Seattle, WA"
    url = "https://nominatim.openstreetmap.org/search?" + urllib.parse.urlencode(
        {"q": query, "format": "json", "limit": 1}
    )
    req = urllib.request.Request(url, headers={"User-Agent": "seattle-crime-notebook"})
    with urllib.request.urlopen(req, timeout=15) as resp:
        results = json.load(resp)
    if not results:
        raise ValueError(f"No geocoding result for {address!r}")
    return float(results[0]["lat"]), float(results[0]["lon"])


def haversine_miles(lat1, lon1, lat2, lon2):
    """Great-circle distance in miles (vectorized)."""
    R = 3958.8  # Earth radius, miles
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def _short(label: str, n: int = 24) -> str:
    return label if len(label) <= n else label[: n - 1] + "…"


def _slug(text: str) -> str:
    """Filesystem-safe filename from a free-form address (handles '/', etc.)."""
    return re.sub(r"[^A-Za-z0-9,.-]+", "_", text).strip("_") or "address"


def _smooth_circular(values, window: int = 3):
    """Centered moving average over a wrap-around (24-hour) sequence."""
    v = np.asarray(values, dtype=float)
    half = window // 2
    ext = np.concatenate([v[-half:], v, v[:half]])
    return np.convolve(ext, np.ones(window) / window, mode="valid")


def analyze(address: str, radius_miles: float = 0.5):
    """Geocode `address`, keep crimes within `radius_miles`, draw + save the dashboard.

    Adds a benchmark/summary line (local rate vs the citywide average for an area
    this size) and a gun-related-incidents-per-year panel. Other rows: crimes per
    year, top offenses (past-year counts + 10/5/1-yr trend), by hour of day
    (5yr vs 1yr overlaid, smoothed), and a 5yr-vs-1yr spatial comparison
    (proportional-symbol locations + per-year density heatmap).
    Exposes `nearby`, `home_lat`, `home_lon` as globals for further exploration.
    """
    global nearby, home_lat, home_lon
    home_lat, home_lon = geocode(address)
    print(f"{address} -> lat {home_lat:.5f}, lon {home_lon:.5f}")

    g = geo.copy()
    g["dist_mi"] = haversine_miles(g["lat"], g["lon"], home_lat, home_lon)
    nearby = g[g["dist_mi"] <= radius_miles].copy()
    print(f"{len(nearby):,} crimes within {radius_miles} mi of {address}")
    if nearby.empty:
        print("No incidents found - try a Seattle address or a larger radius.")
        return nearby

    now = pd.Timestamp.now()
    periods = [("Past 5 years", 5, "#08519c"), ("Past year", 1, "#d95f02")]
    subs = [(label, yrs, color, nearby[nearby["Offense Date"] >= now - pd.DateOffset(years=yrs)])
            for label, yrs, color in periods]
    ncols = len(periods)
    span_years = max((geo["Offense Date"].max() - geo["Offense Date"].min()).days / 365.25, 1e-6)

    # --- Benchmark (c): compare local count to the citywide average for a circle
    # this size, assuming Seattle's overall incident density. Time cancels in the
    # ratio since both sides use the same window.
    circle_area = np.pi * radius_miles ** 2
    expected = len(geo) * circle_area / SEATTLE_LAND_SQMI
    ratio = len(nearby) / expected if expected else float("nan")
    local_per_year = len(nearby) / span_years
    pct_violent = (nearby["Offense Category"] == "VIOLENT CRIME").mean() * 100

    # --- Gun-related incidents (Shooting Type Group is "-" when not applicable) ---
    gun = nearby[nearby["Shooting Type Group"].fillna("-").ne("-")]
    gun_total = len(gun)
    gun_last1 = int((gun["Offense Date"] >= now - pd.DateOffset(years=1)).sum())
    gun_prior_rate = (gun_total - gun_last1) / max(span_years - 1, 1e-6)
    if gun_total == 0:
        gun_note = "no gun-related incidents on record"
    elif gun_last1 >= 2 and gun_last1 > 1.5 * gun_prior_rate:
        gun_note = f"{gun_total} gun-related ({gun_last1} in the last yr - UP vs ~{gun_prior_rate:.1f}/yr)"
    else:
        gun_note = f"{gun_total} gun-related over {span_years:.0f}y (~{gun_total / span_years:.1f}/yr)"

    summary = (f"~{local_per_year:,.0f} incidents/yr within {radius_miles} mi   |   "
               f"{ratio:.1f}x the citywide average for an area this size   |   "
               f"{pct_violent:.0f}% violent   |   {gun_note}")

    # Locations: collapse to unique block points, size by incidents/year (shared scale).
    loc_aggs, rate_max = [], 0.0
    for _, yrs, _c, sub in subs:
        agg = sub.groupby(["lon", "lat"]).size().reset_index(name="cnt")
        agg["rate"] = agg["cnt"] / yrs
        loc_aggs.append(agg)
        rate_max = max(rate_max, float(agg["rate"].max()) if len(agg) else 0.0)
    rate_max = rate_max or 1.0

    pad = 0.1 * ((nearby["lon"].max() - nearby["lon"].min()) or 0.01)
    extent = (nearby["lon"].min() - pad, nearby["lon"].max() + pad,
              nearby["lat"].min() - pad, nearby["lat"].max() + pad)

    fig = plt.figure(figsize=(13, 23), constrained_layout=True)
    fig.suptitle(f"Crime near {address}  (within {radius_miles} mi)",
                 fontsize=16, weight="bold")
    gs = fig.add_gridspec(7, ncols, height_ratios=[0.16, 0.85, 1.0, 0.7, 0.12, 1.2, 1.2])

    # Row 0 (full width): plain-language summary / benchmark.
    sax = fig.add_subplot(gs[0, :]); sax.axis("off")
    sax.text(0.5, 0.5, summary, ha="center", va="center", fontsize=12.5)

    # Row 1: crimes per year split violent vs non-violent (stacked, so the total
    # bar height stays comparable across years) with a % violent line; plus
    # gun-related incidents per year.
    by_year = nearby.groupby(nearby["Offense Date"].dt.year).size()
    vsub = nearby[nearby["Offense Category"] == "VIOLENT CRIME"]
    v_year = vsub.groupby(vsub["Offense Date"].dt.year).size().reindex(by_year.index, fill_value=0)
    nv_year = by_year - v_year
    ax = fig.add_subplot(gs[1, 0])
    ax.bar(by_year.index, v_year.values, color="firebrick", label="violent")
    ax.bar(by_year.index, nv_year.values, bottom=v_year.values, color="steelblue", label="non-violent")
    ax.set_title("Crimes per year (violent vs non-violent; current year partial)", fontsize=11)
    ax.set_xlabel("year"); ax.set_ylabel("count")
    ax2 = ax.twinx()
    pct_v = (v_year / by_year * 100).fillna(0)
    vcolor = "firebrick"  # match the violent bars; line + right axis share this color
    ax2.plot(by_year.index, pct_v.values, color=vcolor, marker="o", lw=1.5, ms=3, label="% violent")
    ax2.set_ylabel("% violent", color=vcolor)
    ax2.tick_params(axis="y", colors=vcolor)
    ax2.spines["right"].set_color(vcolor)
    ax2.set_ylim(0, max(float(pct_v.max()) * 1.5, 1))
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=7, loc="upper left")

    ax = fig.add_subplot(gs[1, 1])
    ax.set_title("Gun-related incidents per year", fontsize=11)
    gun_types = [("Shots Fired (Eyewitness/Casings/Property Damage)", "shots fired", "#fdae6b"),
                 ("Shooting (Non-Fatal Injury)", "non-fatal", "#e6550d"),
                 ("Shooting (Fatal Injury)", "fatal", "#a50f15")]
    if gun_total:
        gy = (gun.assign(yr=gun["Offense Date"].dt.year)
                 .groupby(["yr", "Shooting Type Group"]).size().unstack(fill_value=0)
                 .reindex(by_year.index, fill_value=0))
        bottom = np.zeros(len(gy))
        for full, label, color in gun_types:
            vals = gy[full].to_numpy() if full in gy.columns else np.zeros(len(gy))
            ax.bar(gy.index, vals, bottom=bottom, color=color, label=label)
            bottom += vals
        ax.legend(fontsize=7); ax.set_ylabel("incidents")
    else:
        ax.text(0.5, 0.5, "none within radius", ha="center", va="center", transform=ax.transAxes)
    ax.set_xlabel("year")

    # Row 2: top offenses - LEFT past-year counts, RIGHT avg/year trend by window.
    past_year = nearby[nearby["Offense Date"] >= now - pd.DateOffset(years=1)]
    ranked = (past_year if len(past_year) else nearby)["Offense Sub Category"].value_counts()
    top_cats = ranked.head(10).index.tolist()
    py_counts = past_year["Offense Sub Category"].value_counts().reindex(top_cats).fillna(0)
    trend_windows = [("10 yr", 10, "#bdd7e7"), ("5 yr", 5, "#6baed6"), ("1 yr", 1, "#08519c")]
    trend_rate = {}
    for wlabel, wyears, _c in trend_windows:
        sw = nearby[nearby["Offense Date"] >= now - pd.DateOffset(years=wyears)]
        trend_rate[wlabel] = (sw["Offense Sub Category"].value_counts()
                                .reindex(top_cats).fillna(0) / wyears)
    cats_rev = top_cats[::-1]
    yv = np.arange(len(cats_rev))
    axL = fig.add_subplot(gs[2, 0])
    axL.barh([_short(c) for c in cats_rev], [py_counts[c] for c in cats_rev],
             color="indianred", zorder=3)
    axL.set_title("Top offenses - past year (count)", fontsize=11)
    axL.set_xlabel("incidents"); axL.tick_params(axis="y", labelsize=7)
    axR = fig.add_subplot(gs[2, 1])
    bh = 0.26
    for i, (wlabel, wyears, color) in enumerate(trend_windows):
        axR.barh(yv + (1 - i) * bh, [trend_rate[wlabel][c] for c in cats_rev],
                 height=bh, color=color, label=wlabel, zorder=3)
    axR.set_yticks(yv); axR.set_yticklabels([])
    axR.set_title("Avg incidents/year by window (trend)", fontsize=11)
    axR.set_xlabel("incidents / year"); axR.legend(fontsize=8, title="window")
    # Shade alternating category rows in both panels so the eye can track a row
    # straight across from the labels on the left to the trend bars on the right.
    for _ax in (axL, axR):
        _ax.set_ylim(-0.5, len(cats_rev) - 0.5)
        for _i in range(0, len(cats_rev), 2):
            _ax.axhspan(_i - 0.5, _i + 0.5, color="0.92", zorder=0)

    # Row 3 (full width): by hour of day, both periods overlaid, % of incidents,
    # 3-hour smoothed so a single noisy hour doesn't dominate the shape.
    ax = fig.add_subplot(gs[3, :])
    for label, yrs, color, sub in subs:
        hp = sub.groupby(sub["Offense Date"].dt.hour).size().reindex(range(24)).fillna(0)
        hp = hp / hp.sum() * 100 if hp.sum() else hp
        ax.plot(range(24), _smooth_circular(hp.values), color=color, lw=2,
                label=f"{label} (n={len(sub):,})")
    ax.set_title("By hour of day (% of incidents, 3-hour smoothed)", fontsize=11)
    ax.set_xlabel("hour of day"); ax.set_ylabel("% of incidents")
    ax.set_xticks(range(0, 24, 2)); ax.set_xlim(0, 23); ax.legend(fontsize=9)

    # Row 4: bold column headers for the spatial comparison below.
    for col, (label, yrs, color, sub) in enumerate(subs):
        hax = fig.add_subplot(gs[4, col]); hax.axis("off")
        hax.text(0.5, 0.4, f"{label.upper()}\n(n={len(sub):,})", ha="center",
                 va="center", fontsize=15, fontweight="bold")

    density_axes, hexbins = [], []
    for col, (label, yrs, color, sub) in enumerate(subs):
        # Row 5: proportional-symbol location map (dot area = incidents/year)
        ax = fig.add_subplot(gs[5, col])
        ax.set_title("Locations (dot size = incidents/yr)", fontsize=11)
        agg = loc_aggs[col]
        if len(agg):
            sizes = np.clip(agg["rate"] / rate_max * 170, 4, None)
            ax.scatter(agg["lon"], agg["lat"], s=sizes, alpha=0.45,
                       color="steelblue", edgecolor="none")
        ax.scatter([home_lon], [home_lat], s=140, color="red", marker="*",
                   edgecolor="white", linewidth=0.6)
        ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
        ax.tick_params(labelsize=6)

        # Row 6: per-year density heatmap (light cmap; shared, percentile-clipped scale)
        ax = fig.add_subplot(gs[6, col])
        ax.set_title("Density / year", fontsize=11)
        if not sub.empty:
            hb = ax.hexbin(sub["lon"], sub["lat"], C=np.full(len(sub), 1.0 / yrs),
                           reduce_C_function=np.sum, gridsize=40, extent=extent,
                           cmap="YlOrRd", mincnt=1)
            hexbins.append(hb)
        ax.scatter([home_lon], [home_lat], s=140, color="red", marker="*",
                   edgecolor="black", linewidth=0.6)
        ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
        ax.tick_params(labelsize=6)
        density_axes.append(ax)

    # Shared color scale clipped at the 97th percentile so one hotspot doesn't
    # darken everything; single horizontal colorbar under the row.
    if hexbins:
        allvals = np.concatenate([np.asarray(hb.get_array()) for hb in hexbins])
        vmax = float(np.percentile(allvals, 97)) if allvals.size else 1.0
        vmax = vmax or float(allvals.max() or 1.0)
        for hb in hexbins:
            hb.set_clim(0, vmax)
        fig.colorbar(hexbins[-1], ax=density_axes, location="bottom", shrink=0.6,
                     aspect=40, pad=0.02, extend="max", label="incidents per year")

    charts_dir = Path("charts")
    charts_dir.mkdir(exist_ok=True)
    outfile = charts_dir / (_slug(address) + ".png")
    fig.savefig(outfile, dpi=150, bbox_inches="tight")
    print(f"saved chart -> {outfile}")
    plt.show()
    return nearby

In [ ]:
# --- Run the analysis --------------------------------------------------
# Run this cell, then type or paste a Seattle address at the prompt
# (blank = Space Needle). input() blocks, so "Run All" waits here for you.
# The dashboard renders below and is saved to charts/<address>.png.
ADDRESS = input("Seattle address [Space Needle]: ").strip() or "Space Needle"
RADIUS_MILES = 0.5   # 0.25 = ~4 blocks, 0.5 = ~10 min walk, 1.0 = neighborhood

analyze(ADDRESS, RADIUS_MILES)